# 03 — Playground garde-fous : sqlglot, LIMIT, timeout, logs JSON

**Jalon 2, première partie.** Le serveur refuse déjà beaucoup de choses grâce aux privilèges, à la
transaction en lecture seule et au curseur serveur. Il reste trois trous :

1. une requête de lecture peut appeler des **fonctions dangereuses** (`set_config`, `pg_sleep`…) ou
   lire des **tables système** (`pg_user`, `information_schema`) ;
2. une requête peut être **très coûteuse** et occuper la base ;
3. on ne **trace** rien de ce qui se passe.

On explore ici les outils pour les combler : **sqlglot** (analyse du SQL), `LIMIT` imposé,
`statement_timeout`, et des logs JSON sur stderr.

## 1. sqlglot : du texte SQL à un arbre

sqlglot transforme une requête en **arbre syntaxique** (AST) : chaque nœud est un objet Python
typé (`Select`, `Table`, `Column`, `Count`…). On peut parcourir l'arbre, le modifier, puis le
régénérer en SQL. `read="postgres"` indique le dialecte.

In [ ]:
import os
from pathlib import Path

# get_settings() lit le .env du répertoire courant : on se place à la racine du repo.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import sqlglot
from sqlglot import exp

print("sqlglot", sqlglot.__version__)

arbre = sqlglot.parse_one(
    "SELECT g.name, count(*) AS nb FROM genre g JOIN track t USING (genre_id) GROUP BY g.name",
    read="postgres",
)
print(type(arbre).__name__)
print(repr(arbre)[:600])

On interroge l'arbre avec `find_all`, sans jamais manipuler le texte :

In [ ]:
print("Tables   :", [table.name for table in arbre.find_all(exp.Table)])
print("Colonnes :", [colonne.sql() for colonne in arbre.find_all(exp.Column)])
print("Fonctions:", [type(fonction).__name__ for fonction in arbre.find_all(exp.Func)])
print("Régénéré :", arbre.sql(dialect="postgres"))

### 1.1 Plusieurs instructions, syntaxe inconnue, erreurs
- `sqlglot.parse` (sans `_one`) renvoie **une liste** : une entrée par instruction.
- Ce que sqlglot ne sait pas analyser finement (`SHOW`, `EXPLAIN`) devient un nœud générique
  `Command`, avec un avertissement.
- Une requête invalide lève `ParseError`.

In [ ]:
from sqlglot.errors import ParseError

for requete in ["SELECT 1; DROP TABLE genre", "SHOW search_path", "SELEC 1"]:
    try:
        arbres = sqlglot.parse(requete, read="postgres")
        print(f"{requete!r:32} → {[type(a).__name__ for a in arbres]}")
    except ParseError as erreur:
        print(f"{requete!r:32} → ParseError : {erreur.errors[0]['description']}")

## 2. Règle 1 : une seule instruction, et c'est une lecture

`exp.Query` est la classe commune à `Select` et aux opérations ensemblistes (`Union`…).
Mais la racine ne suffit pas : un `SELECT` peut **contenir** une écriture (CTE `DELETE`), créer
une table (`SELECT … INTO`) ou poser des verrous (`FOR UPDATE`). On cherche donc ces nœuds
**partout** dans l'arbre.

In [ ]:
NOEUDS_INTERDITS = (
    exp.Insert,
    exp.Update,
    exp.Delete,
    exp.Merge,  # écritures
    exp.Into,  # SELECT … INTO nouvelle_table
    exp.Lock,  # FOR UPDATE / FOR SHARE
    exp.Command,
    exp.Create,
    exp.Drop,
    exp.Alter,
    exp.TruncateTable,
)


class RefusSQL(ValueError):
    """Requête refusée par un garde-fou, avec un message destiné à l'agent."""


def analyser(sql: str) -> exp.Query:
    """Règle 1 : une seule instruction, de lecture, sans écriture cachée."""
    try:
        arbres = [a for a in sqlglot.parse(sql, read="postgres") if a is not None]
    except ParseError as erreur:
        raise RefusSQL(f"SQL illisible : {erreur.errors[0]['description']}") from None
    if len(arbres) != 1:
        raise RefusSQL(f"Une seule requête attendue, {len(arbres)} reçue(s).")
    arbre = arbres[0]
    if not isinstance(arbre, exp.Query):
        raise RefusSQL(f"Seule une lecture (SELECT) est acceptée, pas {arbre.key.upper()}.")
    for noeud in arbre.find_all(*NOEUDS_INTERDITS):
        raise RefusSQL(f"Construction interdite : {noeud.key.upper()}.")
    return arbre


def essayer(regle, requetes: list[str]) -> None:
    """Affiche, pour chaque requête, si la règle l'accepte ou la refuse."""
    for requete in requetes:
        try:
            regle(requete)
            print(f"ACCEPTÉE  {requete}")
        except RefusSQL as refus:
            print(f"REFUSÉE   {requete}\n          ↳ {refus}")


essayer(
    analyser,
    [
        "SELECT name FROM genre",
        "SELECT 1 AS x UNION SELECT 2",
        "SELECT 1; DROP TABLE genre",
        "DELETE FROM genre",
        "WITH d AS (DELETE FROM genre RETURNING *) SELECT * FROM d",
        "SELECT * INTO pirate FROM genre",
        "SELECT * FROM genre FOR UPDATE",
    ],
)

## 3. Règle 2 : seulement les tables métier

Le rôle `agent_lecteur` peut lire une partie du catalogue système (`pg_user`, `pg_stat_activity`,
`information_schema`…) : noms des rôles, requêtes en cours, réglages du serveur. Inutile pour
répondre à une question métier, et c'est de l'information sur l'infrastructure.

On n'autorise donc que les tables de la liste blanche (celle de `list_tables`), plus les noms
définis dans la requête elle-même par un `WITH` (CTE).

In [ ]:
from text_to_sql_mcp import db

TABLES_AUTORISEES = set(db.list_table_names())
print(TABLES_AUTORISEES)

In [ ]:
def verifier_tables(arbre: exp.Query) -> None:
    """Règle 2 : uniquement des tables métier (ou des CTE définies dans la requête)."""
    ctes = {cte.alias for cte in arbre.find_all(exp.CTE)}
    for table in arbre.find_all(exp.Table):
        if isinstance(table.this, exp.Func):  # fonction en FROM (generate_series…) : règle 3
            continue
        if table.db and table.db != "public":
            raise RefusSQL(f"Schéma non autorisé : {table.db}.")
        if table.name not in TABLES_AUTORISEES | ctes:
            raise RefusSQL(f"Table non autorisée : {table.name}.")


essayer(
    lambda sql: verifier_tables(analyser(sql)),
    [
        "WITH top AS (SELECT artist_id FROM album) SELECT * FROM top",
        "SELECT * FROM public.genre",
        "SELECT usename FROM pg_user",
        "SELECT * FROM pg_catalog.pg_stat_activity",
        "SELECT * FROM information_schema.tables",
    ],
)

## 4. Règle 3 : pas de fonctions dangereuses

Les fonctions connues de sqlglot deviennent des nœuds typés (`Count`, `Upper`, `TimestampTrunc`…).
Les autres deviennent des nœuds `Anonymous`, avec leur nom. Regardons ce que deviennent quelques
fonctions légitimes et quelques fonctions dangereuses.

In [ ]:
for appel in [
    "count(*)",
    "to_char(invoice_date, 'YYYY')",
    "date_trunc('month', invoice_date)",
    "string_agg(name, ', ')",
    "age(now(), birth_date)",
    "set_config('statement_timeout', '0', false)",
    "pg_sleep(10)",
    "query_to_xml('SELECT 1', true, true, '')",
    "pg_read_file('/etc/passwd')",
]:
    noeud = sqlglot.parse_one(f"SELECT {appel}", read="postgres").find(exp.Func)
    nom = noeud.name if isinstance(noeud, exp.Anonymous) else ""
    print(f"{appel:48} → {type(noeud).__name__} {nom}")

Deux stratégies possibles :

- **Liste noire** : refuser les `Anonymous` dont le nom est dangereux. Simple, mais on peut oublier
  une fonction.
- **Liste blanche** : n'accepter que des fonctions connues. Plus sûr, mais des fonctions
  légitimes comme `age()` deviennent `Anonymous` et seraient refusées : il faudrait les ajouter
  une à une.

Les fonctions d'administration de PostgreSQL commencent presque toutes par `pg_` (`pg_sleep`,
`pg_read_file`, `pg_terminate_backend`…). Une liste noire par **préfixes** couvre donc large.
Attention aussi aux fonctions qui **exécutent du SQL passé en texte**, comme `query_to_xml` :
elles contournent l'analyse de l'arbre, puisque leur SQL n'est qu'une chaîne de caractères.

In [ ]:
PREFIXES_INTERDITS = ("pg_", "dblink", "lo_")
FONCTIONS_INTERDITES = {
    "set_config",
    "current_setting",
    "query_to_xml",
    "query_to_xml_and_xmlschema",
    "cursor_to_xml",
    "table_to_xml",
}


def verifier_fonctions(arbre: exp.Query) -> None:
    """Règle 3 : aucune fonction d'administration ni d'exécution de SQL dynamique."""
    for fonction in arbre.find_all(exp.Anonymous):
        nom = fonction.name.lower()
        if nom.startswith(PREFIXES_INTERDITS) or nom in FONCTIONS_INTERDITES:
            raise RefusSQL(f"Fonction interdite : {nom}.")


essayer(
    lambda sql: verifier_fonctions(analyser(sql)),
    [
        "SELECT first_name, age(now(), birth_date) FROM employee",
        "SELECT set_config('statement_timeout', '0', false)",
        "SELECT name FROM genre WHERE genre_id IN (SELECT 1 WHERE pg_sleep(1) IS NOT NULL)",
        "SELECT query_to_xml('SELECT * FROM pg_user', true, true, '')",
    ],
)

## 5. Règle 4 : imposer un `LIMIT`

On a déjà `fetchmany(cap + 1)` côté Python. Pourquoi un `LIMIT` dans le SQL ? Parce qu'il change
le **travail de la base**. Avec un `ORDER BY` suivi d'un `LIMIT`, PostgreSQL garde seulement les
N meilleures lignes en mémoire (tri « top-N ») au lieu de trier tout le résultat.

Mesurons sur un tri volontairement énorme : un produit cartésien `track × track`, soit environ
12 millions de lignes. On pose un timeout de 5 secondes pour ne pas bloquer le notebook
(partie 6).

In [ ]:
import time

import psycopg
from psycopg.conninfo import make_conninfo

from text_to_sql_mcp.config import get_settings

REQUETE_LOURDE = (
    "SELECT t1.name, t2.name FROM track t1 CROSS JOIN track t2 ORDER BY t1.name, t2.name"
)
CONNINFO_5S = make_conninfo(get_settings().conninfo(), options="-c statement_timeout=5000")


def chronometrer(sql: str) -> None:
    """Exécute la requête via un curseur serveur et lit 101 lignes, comme run_query."""
    debut = time.perf_counter()
    try:
        with psycopg.connect(CONNINFO_5S) as conn, conn.cursor(name="chrono") as cur:
            cur.execute(sql)
            cur.fetchmany(101)
        print(f"{time.perf_counter() - debut:5.2f} s   {sql[-40:]}")
    except psycopg.errors.QueryCanceled:
        print(f"{time.perf_counter() - debut:5.2f} s   ANNULÉE PAR LE TIMEOUT   {sql[-40:]}")


chronometrer(REQUETE_LOURDE + " LIMIT 101")
chronometrer(REQUETE_LOURDE)

Deux leçons :
- le `LIMIT` rend le tri bien moins coûteux : la requête limitée aboutit, l'autre non ;
- même limitée, la requête prend du temps, car il faut bien générer les 12 millions de
  combinaisons. Le `LIMIT` ne remplace pas le **timeout** : les deux couches sont nécessaires.

### 5.1 Comment imposer le `LIMIT` ? Deux options

**Option A, modifier l'arbre** : `arbre.limit(n)`, puis on exécute le SQL **régénéré** par
sqlglot. On garde un `LIMIT` plus petit s'il existe déjà.

In [ ]:
def imposer_limit(arbre: exp.Query, cap: int) -> exp.Query:
    """Règle 4 : LIMIT cap + 1 (le +1 détecte la troncature), sauf LIMIT déjà plus petit."""
    limite = arbre.args.get("limit")
    valeur = limite.expression if limite else None
    if isinstance(valeur, exp.Literal) and valeur.is_int and int(valeur.this) <= cap + 1:
        return arbre
    return arbre.limit(cap + 1)


for requete in [
    "SELECT * FROM genre",
    "SELECT * FROM genre LIMIT 5",
    "SELECT * FROM genre LIMIT 5000",
    "SELECT 1 AS x UNION SELECT 2",
    "SELECT * FROM genre ORDER BY name LIMIT ALL OFFSET 3",
]:
    print(f"{requete:52} → {imposer_limit(analyser(requete), 100).sql(dialect='postgres')}")

Le SQL régénéré n'est pas toujours identique au texte d'origine : sqlglot **normalise**.
C'est équivalent, mais pas mot pour mot :

In [ ]:
for requete in [
    "SELECT date_part('year', invoice_date) FROM invoice",
    "SELECT genre_id::text FROM genre",
    "SELECT date_trunc('month', invoice_date) FROM invoice",
]:
    print(f"{requete:55} → {analyser(requete).sql(dialect='postgres')}")

**Option B, envelopper le texte d'origine** :
`SELECT * FROM (<sql de l'agent>) AS resultat LIMIT n`. Le SQL de l'agent est exécuté tel quel.

Le compromis est subtil, et c'est une vraie question de sécurité :

- **Option A** : on exécute **exactement ce qu'on a validé** (l'arbre). Si sqlglot et PostgreSQL
  comprennent une requête différemment (*parser differential*), c'est la version de sqlglot,
  vérifiée, qui s'exécute. Le prix : une réécriture du SQL, à valider sur le gold set au jalon 3.
- **Option B** : aucune réécriture, mais on exécute un texte que PostgreSQL pourrait comprendre
  autrement que sqlglot. Un attaquant qui trouve une telle différence contourne la validation.
  Les autres couches (privilèges, lecture seule, curseur serveur) limitent alors les dégâts.



## 6. Le timeout : `statement_timeout`

Le timeout se règle **à l'ouverture de la connexion**, via l'option libpq
`options="-c statement_timeout=…"`. Il s'applique alors à chaque requête de la session, sans
requête supplémentaire. Dépassé, PostgreSQL annule la requête : psycopg lève `QueryCanceled`
(SQLSTATE 57014).

Point d'attention pour le code : `QueryCanceled` est une sous-classe d'`OperationalError`, que
`run_query` laisse remonter telle quelle aujourd'hui (comme une panne de connexion). Il faudra
la transformer en message clair pour l'agent.

In [ ]:
CONNINFO_1S = make_conninfo(get_settings().conninfo(), options="-c statement_timeout=1000")

with psycopg.connect(CONNINFO_1S) as conn:
    print("statement_timeout =", conn.execute("SHOW statement_timeout").fetchone()[0])
    try:
        conn.execute("SELECT pg_sleep(2)")
    except psycopg.errors.QueryCanceled as erreur:
        message = erreur.diag.message_primary
        print(f"{type(erreur).__name__} (SQLSTATE {erreur.sqlstate}) : {message}")
        print("Sous-classe d'OperationalError :", isinstance(erreur, psycopg.OperationalError))

### 6.1 Pourquoi bloquer `set_config` (règle 3)
`set_config` peut remettre le timeout à zéro. Dans **la même** requête, c'est trop tard : le
minuteur est armé au début de la requête. Mais il désarme les requêtes **suivantes** de la même
connexion. Deux protections se complètent : la règle 3 refuse `set_config`, et notre connexion
courte n'exécute qu'une requête par appel d'outil.

In [ ]:
with psycopg.connect(CONNINFO_1S, autocommit=True) as conn:
    try:
        conn.execute("SELECT set_config('statement_timeout', '0', false), pg_sleep(2)")
    except psycopg.errors.QueryCanceled:
        print("Même requête : annulée quand même (minuteur déjà armé)")
    conn.execute("SELECT set_config('statement_timeout', '0', false)")
    debut = time.perf_counter()
    conn.execute("SELECT pg_sleep(2)")
    print(f"Requête suivante : aucun timeout, {time.perf_counter() - debut:.1f} s")

## 7. Logs JSON sur stderr

Objectif : une ligne JSON par événement, sur **stderr** (stdout appartient au protocole MCP).
Le JSON se filtre et s'agrège facilement (`jq`, un outil de logs…). Avec le module standard
`logging`, il suffit d'un `Formatter` qui sérialise l'enregistrement, et du paramètre `extra`
pour ajouter des champs structurés.

In [ ]:
import json
import logging
import sys
from datetime import UTC, datetime

CHAMPS_STANDARD = set(logging.makeLogRecord({}).__dict__)

In [ ]:
CHAMPS_STANDARD

In [ ]:
class FormatJSON(logging.Formatter):
    """Une ligne JSON par enregistrement, avec les champs passés via `extra`."""

    def format(self, record: logging.LogRecord) -> str:
        ligne = {
            "horodatage": datetime.fromtimestamp(record.created, UTC).isoformat(),
            "niveau": record.levelname,
            "logger": record.name,
            "evenement": record.getMessage(),
        }
        extras = record.__dict__.items()
        ligne |= {cle: valeur for cle, valeur in extras if cle not in CHAMPS_STANDARD}
        return json.dumps(ligne, ensure_ascii=False, default=str)

In [ ]:
journal = logging.getLogger("playground.garde_fous")
journal.handlers.clear()
gestionnaire = logging.StreamHandler(sys.stderr)
gestionnaire.setFormatter(FormatJSON())
journal.addHandler(gestionnaire)
journal.setLevel(logging.INFO)
journal.propagate = False

journal.info(
    "requete_executee",
    extra={"outil": "run_query", "lignes": 3, "tronque": False, "duree_ms": 12.4},
)
journal.warning(
    "requete_refusee",
    extra={"outil": "run_query", "regle": "fonction", "detail": "pg_sleep"},
)

Règle de sécurité des logs : on logue le SQL (écrit par le modèle, utile au diagnostic), jamais la
chaîne de connexion ni le mot de passe.



## 8. Assemblage et mini red-team

Les quatre règles enchaînées : ce sera le cœur de `guardrails.py`. La fonction renvoie le SQL à
exécuter (option A), ou lève `RefusSQL` avec un message pour l'agent.

In [ ]:
def valider(sql: str, cap: int = 100) -> str:
    """Applique les quatre règles et renvoie le SQL à exécuter."""
    arbre = analyser(sql)
    verifier_tables(arbre)
    verifier_fonctions(arbre)
    return imposer_limit(arbre, cap).sql(dialect="postgres")


TENTATIVES = [
    "SELECT name FROM genre ORDER BY name",
    "SELECT billing_country, sum(total) FROM invoice GROUP BY 1 ORDER BY 2 DESC LIMIT 5",
    "SELECT 1; DROP TABLE genre",
    "WITH d AS (DELETE FROM genre RETURNING *) SELECT * FROM d",
    "SELECT * FROM genre FOR UPDATE",
    "SELECT usename FROM pg_user",
    "SELECT set_config('default_transaction_read_only', 'off', false)",
    "SELECT pg_sleep(60)",
    "SELECT query_to_xml('SELECT * FROM pg_authid', true, true, '')",
    "SELECT name FROM genre WHERE name = 'x' UNION SELECT email FROM customer",
]
for tentative in TENTATIVES:
    try:
        print(f"ACCEPTÉE  {tentative}\n          ↳ exécutée : {valider(tentative)}")
    except RefusSQL as refus:
        print(f"REFUSÉE   {tentative}\n          ↳ {refus}")

Note la dernière ligne : l'`UNION` qui lit les e-mails des clients est **acceptée**. Ce n'est pas
un trou des garde-fous : `customer` est une table métier que l'agent a le droit de lire. Masquer
des colonnes sensibles (e-mail, téléphone) serait une autre politique : des privilèges **par
colonne** sur le rôle (`GRANT SELECT (colonne, …)`). Hors périmètre du MVP, mais bon sujet pour
le README.



## 9. Décisions à prendre pour la suite :

1. **LIMIT : option A (exécuter le SQL régénéré par sqlglot) ou option B (envelopper le texte
   d'origine) ?**
      - Option A retenue.
2. **Fonctions : liste noire par préfixes, ou liste blanche ?**
      - liste noire par préfixe
3. **Tables : liste blanche stricte** (tables métier + CTE), comme en partie 3 ?
      - pour les tables on garde une liste blanche strice
4. **Timeout : quelle valeur** (5 s par défaut, réglable dans `.env` ?) et où : seulement à la
   connexion (`options`), ou aussi sur le rôle PostgreSQL en filet de sécurité
   (`ALTER ROLE … SET statement_timeout`) ?
      - timeout de 30s réglable dans le .env à la connexion et aussi sur le rôle.
5. **Logs :** quels événements tracer côté serveur  ?
      - outil appelé, durée, lignes, refus, erreurs préfixé par le nom de l'agent qui appelle l'outil (pour identifier celui qu'on utilise au cas ou on test un autre modèle)

## 10. Pour aller plus loin (facultatif)

1. Trouve une requête légitime sur Chinook que `valider` refuse à tort.
2. Teste `SELECT * FROM generate_series(1, 100000000)` : quelle règle la laisse passer, et quelle
   couche l'arrête ?
3. Avec DBeaver en admin, lance `GRANT SELECT (customer_id, first_name, last_name, country) ON
   customer TO agent_lecteur` après un `REVOKE SELECT ON customer FROM agent_lecteur`, puis
   observe ce que voit `get_schema` et ce que fait `SELECT *`. (Remets les droits ensuite, ou
   repars de zéro avec `docker compose down -v`.)